# 01 First calls
Example notebook for stage 1. Run each cell in order.

Step 1: set up the kit

In [1]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("KIT_DATA_DIR", ROOT / "data"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-01"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Kit folder found: atp-builder-kit


Step 2: load your key

The key never goes in a cell. `call_llm` reads `GEMINI_API_KEY` from an environment variable, a local `.env` file (git ignores it), or, in Colab, from **Secrets** (the key icon in the left bar: add `GEMINI_API_KEY` and turn on notebook access). The cell below prints only the provider name, never the key.

In [2]:
from kit.llm import call_llm, pick_provider, LLMError
from kit.prices import cost
import pandas as pd

PROVIDER = pick_provider()  # gemini if GEMINI_API_KEY is set, else openai_compatible, mock if KIT_OFFLINE=1
print("Provider:", PROVIDER)
if PROVIDER == "mock":
    print("Warning: this is the offline mock model. Its outputs are not real runs, so do not submit them.")

Provider: mock


Step 3: set the prompt and settings by hand

In [3]:
SYSTEM = "You are the help assistant for Quillnote, a note app. Answer in at most 3 short steps."
TEMPERATURE = 0.2
MAX_OUTPUT_TOKENS = 200
TRUNCATE_CAP = 10  # the last call uses this tiny cap on purpose, to force a truncation
MODEL = None  # None means the provider default (see kit/llm.py)

QUESTIONS = [
    "How do I export a note as PDF?", "My notes stopped syncing", "Can I share a notebook with my team?",
    "How do I undo a deleted note?", "Where is dark mode?", "Can I search inside images?",
    "Why was I charged twice?", "How do I add a tag?", "The app crashes when I open a big note",
    "Can I use Quillnote offline?",
]
CAPS = [MAX_OUTPUT_TOKENS] * (len(QUESTIONS) - 1) + [TRUNCATE_CAP]  # one output cap per call

Step 4: send 10 requests (the 10th with the tiny cap)

In [4]:
results = []
for q, cap in zip(QUESTIONS, CAPS):
    r = call_llm(q, system=SYSTEM, model=MODEL, temperature=TEMPERATURE, max_output_tokens=cap)
    results.append(r)
    print(f"{q[:40]:40} cap {cap:>3} -> {r['text'][:60]!r}")

How do I export a note as PDF?           cap 200 -> 'Mock answer. You asked: How do I export a note as PDF?. Here'
My notes stopped syncing                 cap 200 -> 'Mock answer. You asked: My notes stopped syncing. Here are t'
Can I share a notebook with my team?     cap 200 -> 'Mock answer. You asked: Can I share a notebook with my team?'
How do I undo a deleted note?            cap 200 -> 'Mock answer. You asked: How do I undo a deleted note?. Here '
Where is dark mode?                      cap 200 -> 'Mock answer. You asked: Where is dark mode?. Here are the st'
Can I search inside images?              cap 200 -> 'Mock answer. You asked: Can I search inside images?. Here ar'
Why was I charged twice?                 cap 200 -> 'Mock answer. You asked: Why was I charged twice?. Here are t'
How do I add a tag?                      cap 200 -> 'Mock answer. You asked: How do I add a tag?. Here are the st'
The app crashes when I open a big note   cap 200 -> 'Mock answer. You asked: The

Step 5: the usage table, one row per call

In [5]:
usage = pd.DataFrame([{
    "call": i, "question": q, "output_cap": cap, "input_tokens": r["input_tokens"], "output_tokens": r["output_tokens"],
    "stop_reason": r["stop_reason"], "latency_s": r["latency_s"],
    "cost_usd": cost(r["input_tokens"], r["output_tokens"], r["model"]),
} for i, (q, cap, r) in enumerate(zip(QUESTIONS, CAPS, results), start=1)])
usage.to_csv(OUTPUT / "usage.csv", index=False)
print("Model:", results[0]["model"], "| saved to", OUTPUT.name + "/usage.csv")
usage

Model: mock-model | saved to stage-01/usage.csv


,call,question,output_cap,input_tokens,output_tokens,stop_reason,latency_s,cost_usd
0,1,How do I export a note as PDF?,200,29,29,STOP,0.353,0.000145
1,2,My notes stopped syncing,200,28,27,STOP,0.582,0.000136
2,3,Can I share a notebook with my team?,200,31,30,STOP,0.988,0.000151
3,4,How do I undo a deleted note?,200,29,28,STOP,0.907,0.000141
4,5,Where is dark mode?,200,27,26,STOP,0.218,0.000131
5,6,Can I search inside images?,200,29,28,STOP,0.906,0.000141
6,7,Why was I charged twice?,200,28,27,STOP,0.961,0.000136
7,8,How do I add a tag?,200,27,26,STOP,0.754,0.000131
8,9,The app crashes when I open a big note,200,31,31,STOP,0.360,0.000155
9,10,Can I use Quillnote offline?,10,29,10,MAX_TOKENS,0.874,0.000069


In [6]:
print("Total cost: $%.5f" % usage.cost_usd.sum())
print("Median latency: %.2fs" % usage.latency_s.median())
usage[["input_tokens", "output_tokens"]].describe()

Total cost: $0.00134
Median latency: 0.81s


,input_tokens,output_tokens
count,10.000000,10.000000
mean,28.800000,26.200000
std,1.398412,5.921711
min,27.000000,10.000000
25%,28.000000,26.250000
50%,29.000000,27.500000
75%,29.000000,28.750000
max,31.000000,31.000000


Step 6: find the forced truncation in the table

In [7]:
truncated = usage[usage.stop_reason.astype(str).str.upper().str.contains("MAX")]
print(len(truncated), "row(s) stopped by the output cap (Gemini says MAX_TOKENS, OpenAI style APIs say length):")
print("Text of the cut off answer:", repr(results[-1]["text"]))
truncated

1 row(s) stopped by the output cap (Gemini says MAX_TOKENS, OpenAI style APIs say length):
Text of the cut off answer: 'Mock answer. You asked: Can I use Quilln'


,call,question,output_cap,input_tokens,output_tokens,stop_reason,latency_s,cost_usd
9,10,Can I use Quillnote offline?,10,29,10,MAX_TOKENS,0.874,0.000069


Step 7: trigger a 400 error on purpose

In [8]:
try:
    call_llm(QUESTIONS[0], system=SYSTEM, model=MODEL, temperature=5)  # temperature must be 0 to 2
except LLMError as e:
    print("Caught error. Status:", e.status)
    print("Message:", e.message[:300])

Caught error. Status: 400
Message: Invalid temperature 5: must be between 0 and 2.


Step 8: see what a 429 looks like (simulated)

In [9]:
# A real 429 means you sent too many requests. call_llm retries with longer waits, then gives up.
# This uses the mock provider so it never touches your real quota.
try:
    call_llm("SIMULATE_429", provider="mock")
except LLMError as e:
    print("Caught error. Status:", e.status, "|", e.message)
    print("429 = rate limit. Wait and retry, slow down, or check your quota.")

Rate limited (429). Waiting 0.02s, then try 2 of 4.
Rate limited (429). Waiting 0.04s, then try 3 of 4.
Rate limited (429). Waiting 0.08s, then try 4 of 4.
Caught error. Status: 429 | Resource exhausted: too many requests (simulated).
429 = rate limit. Wait and retry, slow down, or check your quota.


Step 9: next, open the 01 probe set notebook

Your five predicted weaknesses and your short README section go at the end of that notebook, so the whole stage can be read in two notebooks.